In [105]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

In [106]:
INPUT_XLSX  = "processing_data_3.xlsx" 
OUT_XLSX    = "processing_data_4.xlsx"
OUT_PROFILE = "knowledge/cluster_profiles.xlsx"
OUT_README  = "knowledge/README.txt"

In [107]:
# Human-readable meanings for cluster_id (edit to your case)
CLUSTER_LABELS = {
    0: "Low price & common",
    1: "High-end & high rating",
    2: "Mid range & stable sales",
    3: "Outliers / niche products",
}

In [108]:
# =====================
# Helper Functions
# =====================
def price_segment(p):
    try:
        p = float(p)
    except Exception:
        return "Unknown"
    if p < 2_000_000:
        return "Low price"
    elif p < 8_000_000:
        return "Mid range"
    else:
        return "High end"

def rating_tag(r):
    try:
        r = float(r)
    except Exception:
        return "Unknown"
    if r < 3.5:
        return "Low quality"
    elif r < 4.2:
        return "Average quality"
    else:
        return "High quality"

def popularity_tag(q):
    try:
        q = float(q)
    except Exception:
        return "Unknown"
    if q < 1_000:
        return "Low sales"
    elif q < 5_000:
        return "Moderate sales"
    else:
        return "Bestseller"

In [109]:
# =====================
# Load Data
# =====================
in_path = Path(INPUT_XLSX)
if not in_path.exists():
    raise FileNotFoundError(f"Input file not found: {in_path.resolve()}")
df = pd.read_excel(in_path)
df.head()

,visible_impression_info_amplitude_category_l1_name,id,seller_id,name,name_norm,name_key,model_hint,model_norm,model_key,brand,...,pca14,pca15,pca16,pca17,pca18,pca19,pca20,pca21,pca22,pca23
0,Điện Thoại - Máy Tính Bảng,270975124,1,Điện Thoại Oppo A58 6GB/128GB - Hàng Chính Hãng,dien thoai oppo a58 6gb/128gb - hang chinh hang,dien-thoai-oppo-a58-6gb-128gb-hang-chinh-hang,A58,a58,a58,OPPO,...,12.414051,-0.017350,-0.351536,2.378179,1.584658,0.682832,-1.227970,5.198438,1.412686,-0.629606
1,Điện Thoại - Máy Tính Bảng,184059211,1,Apple iPhone 13,apple iphone 13,apple-iphone-13,NaN,NaN,NaN,Apple,...,1.921546,-0.009271,-0.529429,1.393449,0.338010,0.730493,0.951423,0.720119,0.744619,-0.071681
2,Điện Thoại - Máy Tính Bảng,277007015,1,Điện thoại Xiaomi Redmi Note 14 6GB/128GB - Hà...,dien thoai xiaomi redmi note 14 6gb/128gb - ha...,dien-thoai-xiaomi-redmi-note-14-6gb-128gb-hang...,NaN,NaN,NaN,Xiaomi,...,1.642103,-0.890391,-0.165800,0.437587,1.833717,0.714227,1.088240,1.003887,0.211094,0.575695
3,Điện Thoại - Máy Tính Bảng,275510578,1,Điện thoại OPPO A79 5G (8GB/256GB) - Hàng Chín...,dien thoai oppo a79 5g (8gb/256gb) - hang chin...,dien-thoai-oppo-a79-5g-8gb-256gb-hang-chinh-hang,A79,a79,a79,OPPO,...,1.669335,-0.334507,-0.189409,0.182897,-0.190038,0.503656,0.789483,-0.849791,0.088497,0.437558
4,Điện Thoại - Máy Tính Bảng,276515327,1,Điện thoại POCO C75 (6GB/128GB) - Hàng Chính Hãng,dien thoai poco c75 (6gb/128gb) - hang chinh hang,dien-thoai-poco-c75-6gb-128gb-hang-chinh-hang,C75,c75,c75,POCO,...,0.601458,-0.444147,0.136088,0.313957,-0.193857,0.128591,1.250823,-1.047219,0.176688,0.309903


In [110]:
# =====================
# Business Tags & Cluster Meaning
# =====================
df["price_segment"] = df["price"].apply(price_segment) if "price" in df.columns else "Unknown"
df["rating_tag"]    = df["rating_average"].apply(rating_tag) if "rating_average" in df.columns else "Unknown"
df["popularity"]    = df["quantity_sold_value"].apply(popularity_tag) if "quantity_sold_value" in df.columns else "Unknown"

if "cluster_id" in df.columns:
    df["cluster_meaning"] = (
        pd.to_numeric(df["cluster_id"], errors="coerce").map(CLUSTER_LABELS).fillna("Unlabeled cluster")
    )
else:
    df["cluster_meaning"] = "No cluster_id"

df[["price","price_segment","rating_average","rating_tag","quantity_sold_value","popularity","cluster_id","cluster_meaning"]].head(10)

,price,price_segment,rating_average,rating_tag,quantity_sold_value,popularity,cluster_id,cluster_meaning
0,3890000,Mid range,5.0,High quality,25120,Bestseller,3,Outliers / niche products
1,11350000,High end,5.0,High quality,6815,Bestseller,2,Mid range & stable sales
2,3890000,Mid range,5.0,High quality,6256,Bestseller,2,Mid range & stable sales
3,5690000,Mid range,5.0,High quality,5453,Bestseller,2,Mid range & stable sales
4,2290000,Mid range,5.0,High quality,3829,Moderate sales,2,Mid range & stable sales
5,2790000,Mid range,5.0,High quality,2815,Moderate sales,2,Mid range & stable sales
6,5250000,Mid range,5.0,High quality,1937,Moderate sales,2,Mid range & stable sales
7,3990000,Mid range,5.0,High quality,1582,Moderate sales,2,Mid range & stable sales
8,185000,Low price,4.6,High quality,1241,Moderate sales,2,Mid range & stable sales
9,30250000,High end,5.0,High quality,1107,Moderate sales,2,Mid range & stable sales


In [111]:
# =====================
# Cluster Profiles (per cluster_id)
# =====================
profile_cols = [c for c in ["price","rating_average","quantity_sold_value","is_duplicate"] if c in df.columns]
if "cluster_id" in df.columns and profile_cols:
    profiles = (
        df.groupby("cluster_id")[profile_cols]
          .mean(numeric_only=True)
          .reset_index()
          .rename(columns={
            "price": "avg_price",
            "rating_average": "avg_rating",
            "quantity_sold_value": "avg_quantity_sold",
            "is_duplicate": "dup_ratio",
        })
    )
    for need in ["avg_price","avg_rating","avg_quantity_sold","dup_ratio"]:
        if need not in profiles.columns:
            profiles[need] = np.nan
    df = df.merge(profiles, on="cluster_id", how="left")
else:
    profiles = pd.DataFrame(columns=["cluster_id","avg_price","avg_rating","avg_quantity_sold","dup_ratio"])
    for col in ["avg_price","avg_rating","avg_quantity_sold","dup_ratio"]:
        df[col] = np.nan

profiles

,cluster_id,avg_price,avg_rating,avg_quantity_sold,dup_ratio
0,0,4.567446e+06,1.332432,24.108108,0.054054
1,1,1.388209e+07,0.016194,5.141700,0.299595
2,2,3.204549e+06,4.742723,227.887324,0.164319
3,3,3.890000e+06,5.000000,25120.000000,0.000000


In [112]:
# =====================
# Graph-based Enrichment (duplicate links)
# =====================
dup_cols = [c for c in df.columns if c.startswith("dup_id_")]
if "id" in df.columns and dup_cols:
    G = nx.Graph()
    for pid in df["id"].dropna().astype(int).tolist():
        G.add_node(pid)
    for _, row in df.iterrows():
        if pd.isna(row.get("id")):
            continue
        src = int(row["id"])
        for c in dup_cols:
            v = row.get(c)
            if pd.notna(v):
                try:
                    dst = int(v)
                    if src != dst:
                        G.add_edge(src, dst)
                except Exception:
                    pass
    degree_dict = dict(G.degree())
    df["dup_degree"] = df["id"].map(lambda x: degree_dict.get(int(x),0) if pd.notna(x) else 0).astype(int)
    comp_id_map = {}
    for i, comp in enumerate(nx.connected_components(G)):
        for node in comp:
            comp_id_map[node] = i
    df["dup_component_id"] = df["id"].map(lambda x: comp_id_map.get(int(x),-1) if pd.notna(x) else -1).astype(int)
else:
    df["dup_degree"] = 0
    df["dup_component_id"] = -1

df[["id","dup_degree","dup_component_id"]].head(10)


,id,dup_degree,dup_component_id
0,270975124,0,0
1,184059211,0,1
2,277007015,0,2
3,275510578,0,3
4,276515327,1,4
5,278098703,1,4
6,277777809,0,5
7,125182567,0,6
8,35278834,0,7
9,276109904,0,8


In [113]:
df.to_excel(OUT_XLSX, index=False)
profiles.to_excel(OUT_PROFILE, index=False)

with open(OUT_README, "w", encoding="utf-8") as f:
    f.write(
        f"Knowledge-enriched Data artifacts\n"
        f"- Input: {INPUT_XLSX}\n"
        f"- Output: {OUT_XLSX}\n"
        f"- Profiles: {OUT_PROFILE}\n\n"
        "New columns: price_segment, rating_tag, popularity, cluster_meaning, "
        "avg_price/avg_rating/avg_quantity_sold/dup_ratio, "
        "dup_degree, dup_component_id\n"
    )

print("Saved:", OUT_XLSX, ",", OUT_PROFILE, ",", OUT_README)

Saved: processing_data_4.xlsx , knowledge/cluster_profiles.xlsx , knowledge/README.txt
